<a href="https://colab.research.google.com/github/alinerodrigues9446-sys/projeto_sql/blob/main/Controle_de_Vendas_Simples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import sqlite3

# Connect to an in-memory SQLite database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# SQL commands to create tables
create_tables_sql = """
CREATE TABLE clientes (
    id_cliente INTEGER PRIMARY KEY,
    nome VARCHAR(100),
    email VARCHAR(100),
    cidade VARCHAR(50)
);

CREATE TABLE produtos (
    id_produto INTEGER PRIMARY KEY,
    nome VARCHAR(100),
    preco DECIMAL(10,2)
);

CREATE TABLE pedidos (
    id_pedido INTEGER PRIMARY KEY,
    id_cliente INTEGER,
    data_pedido DATE,
    FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente)
);

CREATE TABLE itens_pedido (
    id_item INTEGER PRIMARY KEY,
    id_pedido INTEGER,
    id_produto INTEGER,
    quantidade INTEGER,
    FOREIGN KEY (id_pedido) REFERENCES pedidos(id_pedido),
    FOREIGN KEY (id_produto) REFERENCES produtos(id_produto)
);
"""

# Execute the SQL commands
try:
    cursor.executescript(create_tables_sql)
    conn.commit()
    print("Tables created successfully in an in-memory SQLite database.")
    # You can now use the 'conn' object to interact with the database.
except sqlite3.Error as e:
    print(f"An error occurred: {e}")
finally:
    # It's good practice to close the connection when done
    # For an in-memory database, closing it will delete the data.
    # You might want to keep it open if you plan further operations.
    # conn.close()
    pass


Tables created successfully in an in-memory SQLite database.


In [4]:
insert_data_sql = """
INSERT INTO clientes VALUES
(1, 'Ana Silva', 'ana@email.com', 'São Paulo'),
(2, 'Bruno Costa', 'bruno@email.com', 'Rio de Janeiro'),
(3, 'Carla Souza', 'carla@email.com', 'Belo Horizonte');

INSERT INTO produtos VALUES
(1, 'Notebook', 3500.00),
(2, 'Mouse', 80.00),
(3, 'Teclado', 150.00);

INSERT INTO pedidos VALUES
(1, 1, '2024-01-10'),
(2, 2, '2024-01-15'),
(3, 1, '2024-02-05');

INSERT INTO itens_pedido VALUES
(1, 1, 1, 1),
(2, 1, 2, 2),
(3, 2, 3, 1),
(4, 3, 1, 1);
"""

try:
    cursor.executescript(insert_data_sql)
    conn.commit()
    print("Data inserted successfully.")
except sqlite3.Error as e:
    print(f"An error occurred during data insertion: {e}")


Data inserted successfully.


In [6]:
# Total de vendas por pedido
print("Total de vendas por pedido:")
cursor.execute("""
SELECT p.id_pedido,
       SUM(pr.preco * i.quantidade) AS total_pedido
FROM pedidos p
JOIN itens_pedido i ON p.id_pedido = i.id_pedido
JOIN produtos pr ON i.id_produto = pr.id_produto
GROUP BY p.id_pedido;
""")
for row in cursor.fetchall():
    print(row)

print("\nTotal gasto por cliente:")
# Total gasto por cliente
cursor.execute("""
SELECT c.nome,
       SUM(pr.preco * i.quantidade) AS total_gasto
FROM clientes c
JOIN pedidos p ON c.id_cliente = p.id_cliente
JOIN itens_pedido i ON p.id_pedido = i.id_pedido
JOIN produtos pr ON i.id_produto = pr.id_produto
GROUP BY c.nome;
""")
for row in cursor.fetchall():
    print(row)

print("\nProduto mais vendido (em quantidade):")
# Produto mais vendido (em quantidade)
cursor.execute("""
SELECT pr.nome,
       SUM(i.quantidade) AS total_vendido
FROM produtos pr
JOIN itens_pedido i ON pr.id_produto = i.id_produto
GROUP BY pr.nome
ORDER BY total_vendido DESC;
""")
for row in cursor.fetchall():
    print(row)

print("\nVendas por mês:")
# Vendas por mês
# Note: SQLite does not have a MONTH() function. Using strftime('%m', date_column) instead.
cursor.execute("""
SELECT strftime('%m', p.data_pedido) AS mes,
       SUM(pr.preco * i.quantidade) AS total_vendas
FROM pedidos p
JOIN itens_pedido i ON p.id_pedido = i.id_pedido
JOIN produtos pr ON i.id_produto = pr.id_produto
GROUP BY strftime('%m', p.data_pedido);
""")
for row in cursor.fetchall():
    print(row)


Total de vendas por pedido:
(1, 3660)
(2, 150)
(3, 3500)

Total gasto por cliente:
('Ana Silva', 7160)
('Bruno Costa', 150)

Produto mais vendido (em quantidade):
('Notebook', 2)
('Mouse', 2)
('Teclado', 1)

Vendas por mês:
('01', 3810)
('02', 3500)
